# ChromBP-Net

This notebook will go through the ChromBP net tutorial.

In [ ]:
import os

BILLING_PROJECT_ID = os.environ['WORKSPACE_NAMESPACE']
WORKSPACE = os.environ['WORKSPACE_NAME']
bucket = os.environ['WORKSPACE_BUCKET']
    
print("Billing project: " + BILLING_PROJECT_ID)
print("Workspace: " + WORKSPACE)
print("Bucket: " + bucket)

## Install required dependencies

In [ ]:
! conda install -y -c conda-forge -c bioconda samtools bedtools ucsc-bedgraphtobigwig pybigwig meme

In [ ]:
! pip install chrombpnet pybigwig

## Download example data

In [ ]:
! mkdir -p ~/chrombpnet_tutorial/data/downloads

In [ ]:
# download reference data
!wget https://www.encodeproject.org/files/GRCh38_no_alt_analysis_set_GCA_000001405.15/@@download/GRCh38_no_alt_analysis_set_GCA_000001405.15.fasta.gz -O ~/chrombpnet_tutorial/data/downloads/hg38.fa.gz
!yes n | gunzip ~/chrombpnet_tutorial/data/downloads/hg38.fa.gz

# download reference chromosome sizes 
!wget https://www.encodeproject.org/files/GRCh38_EBV.chrom.sizes/@@download/GRCh38_EBV.chrom.sizes.tsv -O ~/chrombpnet_tutorial/data/downloads/hg38.chrom.sizes

# download reference blacklist regions 
!wget https://www.encodeproject.org/files/ENCFF356LFX/@@download/ENCFF356LFX.bed.gz -O ~/chrombpnet_tutorial/data/downloads/blacklist.bed.gz


In [ ]:
# download bam files

!wget https://www.encodeproject.org/files/ENCFF077FBI/@@download/ENCFF077FBI.bam -O ~/chrombpnet_tutorial/data/downloads/rep1.bam
!wget https://www.encodeproject.org/files/ENCFF128WZG/@@download/ENCFF128WZG.bam -O ~/chrombpnet_tutorial/data/downloads/rep2.bam
!wget https://www.encodeproject.org/files/ENCFF534DCE/@@download/ENCFF534DCE.bam -O ~/chrombpnet_tutorial/data/downloads/rep3.bam


## Preprocessing

### Mapping

In [ ]:
# merge and index bam files
#!samtools merge -@13 -f ~/chrombpnet_tutorial/data/downloads/merged_unsorted.bam ~/chrombpnet_tutorial/data/downloads/rep1.bam  ~/chrombpnet_tutorial/data/downloads/rep2.bam  ~/chrombpnet_tutorial/data/downloads/rep3.bam
print("Merged")
!samtools sort -@13 ~/chrombpnet_tutorial/data/downloads/merged_unsorted.bam -o ~/chrombpnet_tutorial/data/downloads/merged.bam
print("Sorted")
!samtools index -@13 ~/chrombpnet_tutorial/data/downloads/merged.bam
print("Indexed")

### Filtering

### Peak Calling

In [ ]:
# download overlap peaks (default peaks on ENCODE)
!wget https://www.encodeproject.org/files/ENCFF333TAT/@@download/ENCFF333TAT.bed.gz -O ~/chrombpnet_tutorial/data/downloads/overlap.bed.gz

In [ ]:
#Ensure that the peak regions do not intersect with the blacklist regions (extended by 1057 bp on both sides)
!bedtools slop -i ~/chrombpnet_tutorial/data/downloads/blacklist.bed.gz -g ~/chrombpnet_tutorial/data/downloads/hg38.chrom.sizes -b 1057 > ~/chrombpnet_tutorial/data/downloads/temp.bed
!bedtools intersect -v -a ~/chrombpnet_tutorial/data/downloads/overlap.bed.gz -b ~/chrombpnet_tutorial/data/downloads/temp.bed  > ~/chrombpnet_tutorial/data/peaks_no_blacklist.bed

### Define train, validation and test chromosome splits

In [ ]:
!head -n 24  ~/chrombpnet_tutorial/data/downloads/hg38.chrom.sizes >  ~/chrombpnet_tutorial/data/downloads/hg38.chrom.subset.sizes

In [ ]:
!mkdir ~/chrombpnet_tutorial/data/splits
!chrombpnet prep splits -c ~/chrombpnet_tutorial/data/downloads/hg38.chrom.subset.sizes -tcr chr1 chr3 chr6 -vcr chr8 chr20 -op ~/chrombpnet_tutorial/data/splits/fold_0

### Generate non-peaks (Background regions)

In [ ]:
### Generate non-peaks (Background regions)
!chrombpnet prep nonpeaks -g ~/chrombpnet_tutorial/data/downloads/hg38.fa -p ~/chrombpnet_tutorial/data/peaks_no_blacklist.bed -c  ~/chrombpnet_tutorial/data/downloads/hg38.chrom.sizes -fl ~/chrombpnet_tutorial/data/splits/fold_0.json -br ~/chrombpnet_tutorial/data/downloads/blacklist.bed.gz -o ~/chrombpnet_tutorial/data/output

## Training ChromBPNet

First load pretrained bias model for K562

In [ ]:
!mkdir ~/chrombpnet_tutorial/bias_model
!wget https://storage.googleapis.com/chrombpnet_data/input_files/bias_models/ATAC/ENCSR868FGK_bias_fold_0.h5 -O ~/chrombpnet_tutorial/bias_model/ENCSR868FGK_bias_fold_0.h5

In [ ]:
# Train bias-factorized ChromBPNet
! rm -r /home/jupyter/chrombpnet_tutorial/chrombpnet_model/

In [ ]:
! chrombpnet pipeline \
        -ibam ~/chrombpnet_tutorial/data/downloads/merged.bam \
        -d "ATAC" \
        -g ~/chrombpnet_tutorial/data/downloads/hg38.fa \
        -c ~/chrombpnet_tutorial/data/downloads/hg38.chrom.sizes \
        -p ~/chrombpnet_tutorial/data/peaks_no_blacklist.bed \
        -n ~/chrombpnet_tutorial/data/output_negatives.bed \
        -fl ~/chrombpnet_tutorial/data/splits/fold_0.json \
        -b ~/chrombpnet_tutorial/bias_model/ENCSR868FGK_bias_fold_0.h5 \
        -o ~/chrombpnet_tutorial/chrombpnet_model/

## Train Bias model (optional, important for new datasets)

``` python
! chrombpnet bias pipeline \
        -ibam ~/chrombpnet_tutorial/data/downloads/merged.bam \
        -d "ATAC" \
        -g ~/chrombpnet_tutorial/data/downloads/hg38.fa \
        -c ~/chrombpnet_tutorial/data/downloads/hg38.chrom.sizes \
        -p ~/chrombpnet_tutorial/data/peaks_no_blacklist.bed \
        -n ~/chrombpnet_tutorial/data/output_negatives.bed \
        -fl ~/chrombpnet_tutorial/data/splits/fold_0.json \
        -b 0.5 \
        -o ~/chrombpnet_tutorial/bias_model/ \
        -fp k562 \

```

# Generating the outputs

We used the weights of a ChromBPNet model trained to predict patient-specific accessibility tracks in bulk visceral adipose tissue AMSCs.

## Prediction bigwigs

ChromBPNet outputs:

**Bigwig files:**
- Bias-corrected Accessibility profiles
- Bias prediction
- Model predictions without bias correction

**Bigwig files to be visualized as Dynseq tracks in the Epigenomics Browser:**
- Counts contributions
- Profile contributions

In [ ]:
workspace_bucket = 'gs://path_to_workspace_bucket/'
day = 'D0'
patient = '<DONOR_ID>'

model_folder = workspace_bucket + f'chrombpnet_vc/{day}/{patient}/trained_model_factorized_bias/models/'
bm = model_folder + 'bias_model_scaled.h5'
cm = model_folder + 'chrombpnet.h5'
cmb = model_folder + 'chrombpnet_nobias.h5'
regions = workspace_bucket + 'chrombpnet_vc/regions/C5orf67_variant_centered_regions.bed'


In [ ]:
! mkdir -p ~/chrombpnet_vc_results/{day}/{patient}/

! chrombpnet pred_bw \
        -bm {bm} \
        -cm {cm} \
        -cmb {cmb} \
        -r  {regions}\
        -g ~/chrombpnet_tutorial/data/downloads/hg38.fa \
        -c ~/chrombpnet_tutorial/data/downloads/hg38.chrom.sizes \
        -op ~/chrombpnet_vc_results/{day}/{patient}/ \
        #[-b BATCH_SIZE] \
        #[-t TQDM]
        #[-d DEBUG_CHR [DEBUG_CHR ...]]

## Generate contribution score bigwigs

In [ ]:
regions = workspace_bucket + 'gs://path_to_workspace_bucket/chrombpnet_vc/regions/C5orf67_variant_centered_regions.bed' 

!mkdir -p ~/chrombpnet_vc_results/{day}/{patient}/contributions/

! chrombpnet contribs_bw \
        -m gs://path_to_workspace_bucket/chrombpnet_vc/{day}/{patient}/trained_model_factorized_bias/models/chrombpnet_nobias.h5 \
        -r gs://path_to_workspace_bucket/chrombpnet_vc/regions/C5orf67_variant_centered_regions.bed \
        -g ~/chrombpnet_tutorial/data/downloads/hg38.fa \
        -c ~/chrombpnet_tutorial/data/downloads/hg38.chrom.sizes \
        -op ~/chrombpnet_vc_results/{day}/{patient}/contributions/
        #[-pc {counts,profile} [{counts,profile} ...]]
        #[-os OUTPUT_PREFIX_STATS] [-t TQDM]
        #[-d DEBUG_CHR [DEBUG_CHR ...]]

## De Novo Motif Discovery (using modisco)

This is useful for genomewide atac peak analyses.

In [ ]:
! pip install modisco-lite==2.0.7

```python 
modisco motifs  -i H5PY -n MAX_SEQLETS -op OUTPUT_PREFIX [-l N_LEIDEN] [-w WINDOW] [-v]
```

**Modisco report creation:**

```bash
! modisco report \
        -i ~/chrombpnet_vc_data/path_to_workspace_bucket/modisco_vc/modisco_results_profile_scores.h5 \
        -o ~/chrombpnet_vc_results/modisco_report/ \
        -s ~/chrombpnet_vc_results/modisco_report/ \
        -m ~/chrombpnet_vc_data/path_to_workspace_bucket/modisco_vc/JASPAR2024_CORE_non-redundant_pfms_meme.txt
 ```       

## Marginal footprining

## Variant Effect Prediction

In [ ]:
snp_tsv = "gs://path_to_workspace_bucket/chrombpnet_vc/snps/snps.tsv"

! mkdir -p ~/chrombpnet_vc_results/{day}/{patient}/vep/

! chrombpnet snp_score \
        -snps {snp_tsv} \
        -m {cm} \
        -g ~/chrombpnet_tutorial/data/downloads/hg38.fa \
        -op ~/chrombpnet_vc_results/{day}/{patient}/vep/

In [ ]:
! gsutil cp -r  ~/chrombpnet_vc_results/ gs://path_to_workspace_bucket/

# <center> Analyzing variant changes in TF motifs <center>

``` gsutil -m cp -r gs://path_to_chrombpnet_model_bucket/ChromBPNet_input_output//broad/mcl/database/Adipocytes/visceral/ATAC-seq/ChromBPNet/May_24_2021/nonFFA/D8/<DONOR_ID>/trained_model_factorized_bias/ gs://path_to_workspace_bucket/chrombpnet_vc/D8/<DONOR_ID>/ ```

**Unified pipeline:**

In [ ]:
import os

workspace_bucket = 'gs://path_to_workspace_bucket/'
MCL_chromBPNet_bucket = 'gs://path_to_chrombpnet_model_bucket/ChromBPNet_input_output/'


#! gsutil ls {MCL_chromBPNet_bucket}

# Settings:
cell_types = ["AMSC"]
depots = ["Subcutaneous"] # "Visceral"
stim_conds = ["nonFFA"]
days = ['D0', 'D3','D8','D14']
#patients = ['<DONOR_ID_1>','<DONOR_ID_2>'] # Change for different time points

for cell_type in cell_types:
    for depot in depots:
        for stim_cond in stim_conds:
            for day in days:
                patients_dir = MCL_chromBPNet_bucket + f"{cell_type}/{depot}/{stim_cond}/{day}/"
                patients = !gsutil ls {patients_dir}

                for patient_path in patients:
                    patient = patient_path.split('/')[-2]      
                    print(day,patient)
                    # Paths: in the cloud, locally and results path
                    model_bucket = MCL_chromBPNet_bucket + f"{cell_type}/{depot}/{stim_cond}/{day}/{patient}/trained_model_factorized_bias/models/"
                    
                    ! gsutil ls {model_bucket}
                    
                    local_model_folder = '~/local_models/'
                    results_path = f"~/chrombpnet_results/{cell_type}/{depot}/{stim_cond}/{day}/{patient}/"
                    
                    ! mkdir -p {local_model_folder}
                    ! ls {local_model_folder}
                    
                    # Copy the required files locally:
                    ! gsutil cp -r {model_bucket}* {local_model_folder}
                    ! ls {local_model_folder}
                    
                    #Create results path
                    ! mkdir -p {results_path}
                    ! mkdir -p {results_path}predictions/
                    ! mkdir -p {results_path}contributions/
                    ! mkdir -p {results_path}vep/
                    
                    # Model paths
                    bm = local_model_folder + 'bias_model_scaled.h5'
                    cm = local_model_folder + 'chrombpnet.h5'
                    cmb = local_model_folder + 'chrombpnet_nobias.h5'
                    regions = workspace_bucket + 'data/regions/regions_combined.bed' 
                    snps = workspace_bucket + 'data/snps/snps_combined.tsv' 

                    # Get ATAC predictions
                    ! chrombpnet pred_bw \
                            -bm {bm} \
                            -cm {cm} \
                            -cmb {cmb} \
                            -r  {regions}\
                            -g ~/chrombpnet_tutorial/data/downloads/hg38.fa \
                            -c ~/chrombpnet_tutorial/data/downloads/hg38.chrom.sizes \
                            -op {results_path}predictions/ \
                            #[-b BATCH_SIZE] \
                            #[-t TQDM]
                            #[-d DEBUG_CHR [DEBUG_CHR ...]]

                    # Get contributions towards the profiles
                    ! chrombpnet contribs_bw \
                            -m {cmb} \
                            -r {regions} \
                            -g ~/chrombpnet_tutorial/data/downloads/hg38.fa \
                            -c ~/chrombpnet_tutorial/data/downloads/hg38.chrom.sizes \
                            -op {results_path}contributions/
                            #[-pc {counts,profile} [{counts,profile} ...]]
                            #[-os OUTPUT_PREFIX_STATS] [-t TQDM]
                            #[-d DEBUG_CHR [DEBUG_CHR ...]]

                    
                    # Variant effect prediction (VEP)
                    ! chrombpnet snp_score \
                            -snps {snps} \
                            -m {cm} \
                            -g ~/chrombpnet_tutorial/data/downloads/hg38.fa \
                            -op {results_path}vep/
                    
                    # Remove local models:
                    ! rm -r {local_model_folder}





In [ ]:
#! gsutil -m cp -r ~/chrombpnet_results/ gs://path_to_chrombpnet_model_bucket/ChromBPNet_input_output/AMSC/